# Stage 1: base retention and bf16 drift

## Setup

In [ ]:
import os

if not os.path.isdir("/content/kl-reseach"):
    !git clone https://github.com/JuaniLlaberia/kl-reseach.git /content/kl-reseach
%cd /content/kl-reseach
!git pull --ff-only
!pip install -q -r requirements.txt
!pip uninstall -q -y torchaudio torchvision


## Check the environment

In [ ]:
%cd /content/kl-reseach
import importlib.metadata as metadata
import os
import re

import torch

os.environ["HF_ALLOW_CODE_EVAL"] = "1"

with open("requirements.txt") as file:
    pins = dict(re.sub(r"\[.*\]", "", line.strip()).split("==") for line in file if "==" in line)
installed = {name: metadata.version(name).split("+")[0] for name in pins}
wrong = {name: (installed[name], pins[name]) for name in pins if installed[name] != pins[name]}

print(installed)
print("langdetect", metadata.version("langdetect"))
if wrong:
    raise RuntimeError(f"Not on the pinned versions (installed, pinned): {wrong}")
if torch.__version__.split("+")[0] != pins["torch"]:
    raise RuntimeError(f"torch {torch.__version__} is loaded. Restart the session.")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU available.")

from transformers import Qwen2ForCausalLM

print("GPU:", torch.cuda.get_device_name(0), "| bf16 supported:", torch.cuda.is_bf16_supported())

/content/kl-reseach
{'torch': '2.15.0', 'transformers': '5.22.1', 'trl': '1.17.1', 'datasets': '5.1.0', 'accelerate': '1.15.0', 'bitsandbytes': '0.50.2', 'lm-eval': '0.4.13', 'wandb': '0.31.0', 'python-dotenv': '1.2.4', 'pytest': '9.2.0'}
langdetect 1.0.9
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition | bf16 supported: True


## Results folder

Files in `/content` are lost when the runtime is recycled, so results go to Drive. Copy them to `results/qwen05_base/` in the repo afterwards.

In [3]:
from google.colab import drive

drive.mount("/content/drive")
RESULTS_ROOT = "/content/drive/MyDrive/kl-reseach"

Mounted at /content/drive


## Retention suite

In [ ]:
import gc
import subprocess
import time
from pathlib import Path

import lm_eval
import numpy as np
import transformers
from lm_eval.models.huggingface import HFLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"
EVAL_RUNS = [
    {"task": "hellaswag", "num_fewshot": 0, "batch_size": 8},
    {"task": "truthfulqa_mc2", "num_fewshot": 0, "batch_size": 8},
    {"task": "winogrande", "num_fewshot": 5, "batch_size": 8},
    {"task": "mmlu", "num_fewshot": 5, "batch_size": 8},
    {"task": "ifeval", "num_fewshot": 0, "batch_size": 8},
    {"task": "humaneval_instruct", "num_fewshot": 0, "batch_size": 8},
]
DTYPE = "float32"
CHAT_TEMPLATE = True
DEVICE = "cuda:0"
GEN_KWARGS = {"repetition_penalty": 1.0}

In [5]:
def bootstrap_ci(values: list[float], iters: int = 10000, seed: int = 0) -> list[float]:
    """
    Computes a percentile bootstrap CI of the mean. lm-eval only reports a closed-form standard error for it.

    Args:
        values (list[float]): Score of each sample.
        iters (int): Number of resamples.
        seed (int): Seed of the resampling.
    Returns:
        The lower and upper bounds of the 95% CI.
    """
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    means = [values[rng.integers(len(values), size=len(values))].mean() for _ in range(iters)]
    return [float(bound) for bound in np.percentile(means, [2.5, 97.5])]

def summarize_samples(samples: dict[str, list[dict]]) -> tuple[dict, list[dict]]:
    """
    Computes the score and CI of each metric, pooling subtasks (MMLU's subjects) so each question weighs the same.

    Args:
        samples (dict[str, list[dict]]): Per-sample output of lm-eval, by subtask.
    Returns:
        The score and CI of each metric, and one row per sample without its prompt.
    """
    rows = []
    for subtask, task_samples in samples.items():
        for sample in task_samples:
            row = {"subtask": subtask, "doc_id": sample["doc_id"]}
            # Aggregate-only metrics, like IFEval's instruction-level scores, aren't one number per sample
            row.update({m: sample[m] for m in sample["metrics"] if isinstance(sample[m], (bool, int, float))})
            response = sample["resps"][0][0]
            if isinstance(response, str):
                row["response"] = response
            rows.append(row)

    metrics = [name for name in rows[0] if name not in ("subtask", "doc_id", "response")]
    scores = {}
    for metric in metrics:
        values = [float(row[metric]) for row in rows]
        scores[metric] = {"mean": float(np.mean(values)), "ci95": bootstrap_ci(values), "n": len(values)}

    return scores, rows

def write_json(path: Path, content: dict) -> None:
    """
    Writes a JSON file in one step, so a crash can't leave it half written.

    Args:
        path (Path): Path of the file.
        content (dict): Content of the file.
    Returns:
        None.
    """
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(content, indent=2, default=str), encoding="utf-8")
    temporary.replace(path)

def run_retention(model_path: str, out_dir: str, run: str, checkpoint: str, revision: str | None = None, limit: int | None = None) -> dict:
    """
    Runs the retention suite on one checkpoint. Saves after each benchmark and skips the ones already saved.

    Args:
        model_path (str): Hugging Face model name or local checkpoint path.
        out_dir (str): Folder of this run's results.
        run (str): Run name.
        checkpoint (str): Checkpoint name, which names the results file.
        revision (str | None): Hugging Face commit of the model, if it has one.
        limit (int | None): Documents per task, for smoke tests only. Never for reported scores.
    Returns:
        The content of the results file.
    """
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{checkpoint}.json"

    setup = {
        "run": run,
        "checkpoint": checkpoint,
        "model": model_path,
        "revision": revision,
        "dtype": DTYPE,
        "chat_template": CHAT_TEMPLATE,
        "gen_kwargs": GEN_KWARGS,
        "eval_runs": EVAL_RUNS,
        "limit": limit,
        "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
        "harness_version": lm_eval.__version__,
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else str(DEVICE),
    }

    out = {**setup, "retention": {}}
    if out_path.exists():
        out = json.loads(out_path.read_text(encoding="utf-8"))
        different = [key for key, value in setup.items() if out.get(key) != value]
        if different:
            raise ValueError(f"{out_path} was written with a different setup ({different}), so it can't be resumed.")

    lm = HFLM(pretrained=model_path, revision=revision, dtype=DTYPE, batch_size=EVAL_RUNS[0]["batch_size"], device=DEVICE)

    for eval_run in EVAL_RUNS:
        task = eval_run["task"]
        if task in out["retention"]:
            print(f"{task}: already saved, skipping")
            continue

        lm.batch_size_per_gpu = eval_run["batch_size"]
        start = time.perf_counter()
        res = lm_eval.simple_evaluate(
            model=lm,
            tasks=[task],
            num_fewshot=eval_run["num_fewshot"],
            apply_chat_template=CHAT_TEMPLATE,
            fewshot_as_multiturn=CHAT_TEMPLATE and eval_run["num_fewshot"] > 0,
            gen_kwargs=GEN_KWARGS,
            random_seed=0,
            numpy_random_seed=0,
            torch_random_seed=0,
            fewshot_random_seed=0,
            confirm_run_unsafe_code=True,
            log_samples=True,
            limit=limit,
        )
        minutes = (time.perf_counter() - start) / 60

        scores, rows = summarize_samples(res["samples"])
        with open(out_dir / f"{checkpoint}_{task}_samples.jsonl", "w", encoding="utf-8") as file:
            for row in rows:
                file.write(json.dumps(row) + "\n")

        out["retention"][task] = {
            "minutes": minutes,
            "scores": scores,
            "harness_results": res["results"],
            "harness_groups": res.get("groups", {}),
        }
        write_json(out_path, out)
        print(f"{task}: {minutes:.1f} min | saved")

    del lm
    gc.collect()
    torch.cuda.empty_cache()
    return out

### Smoke test

Catches import, HumanEval and saving errors in a few minutes. These scores aren't usable, and they go in their own folder.

In [ ]:
smoke = run_retention(MODEL_NAME, f"{RESULTS_ROOT}/smoke/qwen05_base", "qwen05_base", "base", MODEL_REVISION, limit=2)
print(list(smoke["retention"]))

### Official base run

If it crashes, rerun this cell: benchmarks already saved are skipped. Disconnect the runtime when you're done.

In [7]:
results = run_retention(MODEL_NAME, f"{RESULTS_ROOT}/results/qwen05_base", "qwen05_base", "base", MODEL_REVISION)

for task, result in results["retention"].items():
    for metric, score in result["scores"].items():
        print(f"{task:20s} {metric:28s} {score['mean']:.4f} CI {score['ci95'][0]:.4f}-{score['ci95'][1]:.4f} n={score['n']} | {result['minutes']:.1f} min")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Running loglikelihood requests: 100%|██████████| 40168/40168 [02:10<00:00, 308.13it/s]


hellaswag: 2.5 min | saved


Running loglikelihood requests: 100%|██████████| 5882/5882 [00:30<00:00, 190.75it/s]


truthfulqa_mc2: 0.6 min | saved


Running loglikelihood requests: 100%|██████████| 2534/2534 [00:13<00:00, 190.21it/s]


... (3 outputs left out)

Running generate_until requests: 100%|██████████| 541/541 [11:42<00:00,  1.30s/it]


ifeval: 11.8 min | saved


Running generate_until requests: 100%|██████████| 164/164 [01:02<00:00,  2.61it/s]


humaneval_instruct: 1.5 min | saved
hellaswag            acc                          0.3971 CI 0.3878-0.4067 n=10042 | 2.5 min
hellaswag            acc_norm                     0.4830 CI 0.4731-0.4928 n=10042 | 2.5 min
truthfulqa_mc2       acc                          0.4535 CI 0.4242-0.4826 n=817 | 0.6 min
winogrande           acc                          0.5478 CI 0.5201-0.5754 n=1267 | 0.3 min
mmlu                 acc                          0.4595 CI 0.4512-0.4680 n=14042 | 7.0 min
ifeval               prompt_level_strict_acc      0.2588 CI 0.2218-0.2976 n=541 | 11.8 min
ifeval               prompt_level_loose_acc       0.2810 CI 0.2440-0.3198 n=541 | 11.8 min
humaneval_instruct   pass@1                       0.3354 CI 0.2622-0.4085 n=164 | 1.5 min


## bf16 drift

Uses Qwen with noise added to its weights, as there's no fine-tuned checkpoint yet. Repeat on a smoke-run checkpoint in Stage 2.

In [ ]:
import copy
import json
from pathlib import Path

import torch
from datasets import load_dataset
from transformers import AutoTokenizer

from src.kl.measure import load_model, measure_kl
from src.kl.sampling import build_prompt_ids, sample_responses

OUT = Path(f"{RESULTS_ROOT}/drift")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
rows = load_dataset("cais/mmlu", "all", split="test").shuffle(seed=0).select(range(100))
prompts = build_prompt_ids(tokenizer, [{"prompt_id": i, "text": r["question"]} for i, r in enumerate(rows)])

base = load_model(MODEL_NAME, torch.float32, DEVICE, MODEL_REVISION)
base_path = OUT / "base.jsonl"
sample_responses(base, tokenizer, prompts, base_path, 7, 16, 512, {"model": MODEL_NAME, "revision": MODEL_REVISION})
bf16_base = copy.deepcopy(base).to(torch.bfloat16)

drift = {}
for noise in (0.01, 0.05):
    torch.manual_seed(1)
    ft = copy.deepcopy(base)
    with torch.no_grad():
        for parameter in ft.parameters():
            parameter.add_(torch.randn_like(parameter) * parameter.std() * noise)

    ft_path = OUT / f"ft_{noise}.jsonl"
    sample_responses(ft, tokenizer, prompts, ft_path, 11, 16, 512, {"model": MODEL_NAME, "revision": MODEL_REVISION, "noise": noise})

    fp32, _ = measure_kl(base, ft, base_path, ft_path, 8)
    bf16, _ = measure_kl(bf16_base, copy.deepcopy(ft).to(torch.bfloat16), base_path, ft_path, 8, allow_bf16=True)
    drift[str(noise)] = {name: {"fp32": fp32[name], "bf16": bf16[name]} for name in fp32}

    print(f"noise={noise}")
    for name in fp32:
        difference = abs(fp32[name] - bf16[name])
        print(f"{name:40s} fp32={fp32[name]:.6e} bf16={bf16[name]:.6e} diff={difference:.2e} relative={difference / abs(fp32[name]):.2%}")

(OUT / "drift.json").write_text(json.dumps(drift, indent=2), encoding="utf-8")